# Notebook 01: Download and Process Consumer Expenditure Data

## Overview

This notebook downloads **Consumer Expenditure Survey (CE)** data directly from the **U.S. Bureau of Labor Statistics (BLS)**.

The downloaded tables include:

- Aggregate annual consumer expenditures across multiple spending categories for the United States.
- The percentage distribution of expenditures by household income group.
- The number of consumer units (households) represented in each income group.

Using these data, this notebook calculates the **average annual grocery expenditure per household** for each income group. These estimates serve as the baseline household spending values used throughout the remainder of this project to estimate grocery demand at the census tract level.

## Output

The final output of this notebook is a table containing the estimated **average annual grocery expenditure per household** by income group, which will be used as an input to the retail demand model in subsequent notebooks.

In [2]:
import pandas as pd
from pathlib import Path

In [3]:
project_root = Path.cwd()
if not Path("data").exists():
    project_root = project_root.parent          

print(project_root)

d:\Wu\2026\Project Portfolio\003 Project\grocery-demand-estimation


In [4]:
data_file = Path(project_root/ "data/raw/cu_income.csv")
data_file.exists()

True

In [5]:
ce_data = pd.read_csv(data_file, header=None)

In [6]:
ce_data.head(10)

,0,1,2,3,4,5,6,7,8
0,Table 1101. Quintiles of income before taxes: ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"(Aggregates in millions of dollars, unless oth...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Item,Aggregate,Lowest\n20\npercent,Second\n20\npercent,Third\n20\npercent,Fourth\n20\npercent,Highest\n20\npercent,NaN,NaN
4,Number of consumer units (in thousands) a/,"135,760","27,139","27,186","26,959","27,205","27,272",NaN,NaN
5,Percent distribution of consumer units,100.0,20.0,20.0,19.9,20.0,20.1,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Consumer unit characteristics (mean values):,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Income before taxes,"$104,207","$16,658","$42,925","$74,474","$121,548","$264,510",NaN,NaN


In [7]:
ce_selected = ce_data.iloc[[3,4,5, 9,51], 0:7]
ce_selected.head()

,0,1,2,3,4,5,6
3,Item,Aggregate,Lowest\n20\npercent,Second\n20\npercent,Third\n20\npercent,Fourth\n20\npercent,Highest\n20\npercent
4,Number of consumer units (in thousands) a/,"135,760","27,139","27,186","26,959","27,205","27,272"
5,Percent distribution of consumer units,100.0,20.0,20.0,19.9,20.0,20.1
9,Income before taxes,"$104,207","$16,658","$42,925","$74,474","$121,548","$264,510"
51,Food at home,"842,257",12.3,15.9,18.7,23.0,30.0


In [8]:
# set column names and row index 
ce = ce_selected.copy()

ce.columns = ce.iloc[0]
ce = ce.iloc[1:]
ce.columns.name = None

ce = ce.set_index(ce.columns[0])
ce.index.name = None

ce

,Aggregate,Lowest\n20\npercent,Second\n20\npercent,Third\n20\npercent,Fourth\n20\npercent,Highest\n20\npercent
Number of consumer units (in thousands) a/,"135,760","27,139","27,186","26,959","27,205","27,272"
Percent distribution of consumer units,100.0,20.0,20.0,19.9,20.0,20.1
Income before taxes,"$104,207","$16,658","$42,925","$74,474","$121,548","$264,510"
Food at home,"842,257",12.3,15.9,18.7,23.0,30.0


In [9]:
ce.columns

Index(['Aggregate', 'Lowest\n20\npercent', 'Second\n20\npercent',
       'Third\n20\npercent', 'Fourth\n20\npercent', 'Highest\n20\npercent'],
      dtype='str')

In [10]:
ce.index

Index(['Number of consumer units (in thousands) a/ ',
       'Percent distribution of consumer units', 'Income before taxes',
       'Food at home'],
      dtype='str')

In [11]:
# rename columns and rows
ce = ce.rename(columns={
    "Lowest\n20\npercent": "lowest20pct",
    "Second\n20\npercent": "second20pct",
    "Third\n20\npercent": "third20pct",
    "Fourth\n20\npercent": "fourth20pct",
    "Highest\n20\npercent": "highest20pct"
})

ce = ce.rename(index={
    'Number of consumer units (in thousands) a/ ': 'num_cu',
    'Percent distribution of consumer units': 'pct_cu',
    'Income before taxes': 'income_mean',
    'Food at home': 'grocery'
})
ce

,Aggregate,lowest20pct,second20pct,third20pct,fourth20pct,highest20pct
num_cu,"135,760","27,139","27,186","26,959","27,205","27,272"
pct_cu,100.0,20.0,20.0,19.9,20.0,20.1
income_mean,"$104,207","$16,658","$42,925","$74,474","$121,548","$264,510"
grocery,"842,257",12.3,15.9,18.7,23.0,30.0


## Calculate two statistics: group grocery expenditure, unit grocery expenditure
#### group grocery expenditure = aggregated grocery expenditure * expenditure share / 100
#### average grocery expenditure per consumer unit = 1000 * group grocery expenditure / number of consumer unites in each group 

In [12]:
ce = (
    ce.replace(r"[$,]", "", regex=True)    # regular expression 
    .apply(pd.to_numeric, errors="coerce")
)
ce.dtypes

Aggregate       float64
lowest20pct     float64
second20pct     float64
third20pct      float64
fourth20pct     float64
highest20pct    float64
dtype: object

In [13]:
ce_t = ce.T
ce_t['grocery_pct'] = ce_t['grocery']
ce_t.loc['Aggregate','grocery_pct'] = 100
ce_t

,num_cu,pct_cu,income_mean,grocery,grocery_pct
Aggregate,135760.0,100.0,104207.0,842257.0,100.0
lowest20pct,27139.0,20.0,16658.0,12.3,12.3
second20pct,27186.0,20.0,42925.0,15.9,15.9
third20pct,26959.0,19.9,74474.0,18.7,18.7
fourth20pct,27205.0,20.0,121548.0,23.0,23.0
highest20pct,27272.0,20.1,264510.0,30.0,30.0


In [14]:
ce_t["grocery_group"] = ce_t.loc["Aggregate","grocery"] * ce_t["grocery_pct"] / 100
ce_t["avg_grocery"] = 1000 * ce_t["grocery_group"] / ce_t["num_cu"]

ce_t

,num_cu,pct_cu,income_mean,grocery,grocery_pct,grocery_group,avg_grocery
Aggregate,135760.0,100.0,104207.0,842257.0,100.0,842257.000,6204.014437
lowest20pct,27139.0,20.0,16658.0,12.3,12.3,103597.611,3817.296547
second20pct,27186.0,20.0,42925.0,15.9,15.9,133918.863,4926.023063
third20pct,26959.0,19.9,74474.0,18.7,18.7,157502.059,5842.281205
fourth20pct,27205.0,20.0,121548.0,23.0,23.0,193719.110,7120.717148
highest20pct,27272.0,20.1,264510.0,30.0,30.0,252677.100,9265.074069


### Insert a column of the lowest average income for each income group for reference
### These numbers are directly from Bureau of Labor Statistics based on nationwide population. 

In [15]:
ce_t["highest_income"] = [9999999, 29932, 57452, 94511, 155925, 9999999]
ce_t

,num_cu,pct_cu,income_mean,grocery,grocery_pct,grocery_group,avg_grocery,highest_income
Aggregate,135760.0,100.0,104207.0,842257.0,100.0,842257.000,6204.014437,9999999
lowest20pct,27139.0,20.0,16658.0,12.3,12.3,103597.611,3817.296547,29932
second20pct,27186.0,20.0,42925.0,15.9,15.9,133918.863,4926.023063,57452
third20pct,26959.0,19.9,74474.0,18.7,18.7,157502.059,5842.281205,94511
fourth20pct,27205.0,20.0,121548.0,23.0,23.0,193719.110,7120.717148,155925
highest20pct,27272.0,20.1,264510.0,30.0,30.0,252677.100,9265.074069,9999999


In [16]:
# save table
ce_t.to_csv(project_root/"data/processed/avg_expenditure_by_income.csv", index=True)